# Initial results of creating knowledge graphs for research papers

## Steps 1 & 2: Pip Installs And Imports

In [1]:
# Step 1: Do pip installs
packages = ["networkx", "pyvis", "PyMuPDF",
            "requests", "spacy", "regex",
            "keybert", "transformers", "sentence-transformers",
            "nltk"]

def install(package: str):
  !uv pip install -q {package}

for package in packages:
  install(package)

print("Installs finished!")

Installs finished!


In [2]:
# Step 2: All necessary imports
import networkx as nx
from pyvis.network import Network
import fitz # PyMuPDF
import requests
import spacy
import itertools
from collections import Counter
import re
from keybert import KeyBERT

from transformers import pipeline
from itertools import combinations
from collections import defaultdict

from sentence_transformers import SentenceTransformer, util
import numpy as np

from IPython.display import HTML

In [3]:
# Download spaCy model only if not already installed
import spacy.util
model_name = 'en_core_web_sm'
if not spacy.util.is_package(model_name):
    # This will always get the latest version
    !uv pip install -q spacy
    !python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------------------------------------- 0.0/12.8 MB 279.3 kB/s eta 0:00:46
     --------------------------------------- 0.1/12.8 MB 774.0 kB/s eta 0:00:17
     -- ------------------------------------- 0.8/12.8 MB 3.7 MB/s eta 0:00:04
     --------- ------------------------------ 3.1/12.8 MB 12.4 MB/s eta 0:00:01
     ------------------ --------------------- 6.0/12.8 MB 19.1 MB/s eta 0:00:01
     --------------------------- ------------ 8.8/12.8 MB 24.5 MB/s eta 0:00:01
     ------------------------------------- - 12.3/12.8 MB 65.6 MB/s eta 0:00:01
     --------------------------------------  12.8/12.8 MB 65.2 MB/s eta 0:00:01
     --------------------------------------  12.8/12.8 MB 65.2 MB/s eta 0:00:01
     --------------------------------------  12.8/12.8 MB 65.2 MB


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 3: Text Extraction

In [4]:
def pdf_to_text(path: str) -> str:
    with fitz.open(path) as doc:
        return "\n".join(page.get_text() for page in doc)

def load_paper_text_from_url(paper_url: str, paper_filename: str) -> str:
  response = requests.get(paper_url)
  with open(paper_filename, "wb") as f:
    f.write(response.content)

  return pdf_to_text(paper_filename)

In [5]:
try:
    import google.colab
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Not running in Google Colab. Skipping file upload.")

def load_paper_text_from_file(paper_path: str):
  doc = fitz.open(paper_path)
  text = ""
  for page in doc:
      text += page.get_text()
  return text

Not running in Google Colab. Skipping file upload.


In [6]:
# paper_url = "https://zhenlab.com/wp-content/uploads/2025/02/Transfer_learning_paper___Bioinformatics_Advances.pdf"
# paper_filename = "zhenlab_paper.pdf"
# paper_text = load_paper_text_from_url(paper_url, paper_filename)

# paper_path = "Can MOOC Instructor Be Portrayed by Semantic Features.pdf"
# paper_text = load_paper_text_from_file(paper_path)
# print(f"Sample text from paper:\n{paper_text[:500]}...")

## Step 4: Creation of Nodes

In [7]:
# Step 4: Creates nodes
# Note this is a section of the code where there are multiple strategies
# to do this step. Some examples include:
#     - Named-Entity Recognition
#     - Extract noun phrases
#     - Extract ranked phrases with KeyBERT

def baseline_create_nodes(paper_text:str, node_limit: int) -> list:
  # The following does noun phrase extraction
  nlp = spacy.load("en_core_web_sm")
  doc = nlp(paper_text)
  noun_phrases = [chunk.text.lower().strip() for chunk in doc.noun_chunks]

  # Reduce number of nodes, required for successful rendering
  noun_phrases = [np for np in noun_phrases if len(np.split()) > 1 and len(np) > 3]
  phrase_counts = Counter(noun_phrases)

  filtered_phrases = [phrase for phrase, _ in phrase_counts.most_common(node_limit)]
  nodes = list(set(filtered_phrases))
  # print(len(nodes))

  return nodes

In [8]:
# More summary focused method of creating nodes

def extract_sections(text):
    section_keywords = ['abstract', 'introduction', 'background', 'related work', 'method', 'methods',
                        'materials', 'experiments', 'results', 'discussion', 'conclusion', 'conclusions'] # won't put references, extra gibberish
    section_pattern = r'(?i)^(' + '|'.join(re.escape(k) for k in section_keywords) + r')$'

    lines = text.split('\n')
    sections = {}
    current_section = None
    for line in lines:
        stripped = line.strip().lower()
        if re.match(section_pattern, stripped):
            current_section = stripped
            print("Found section: " + current_section)
            sections[current_section] = []
        elif current_section:
            sections[current_section].append(stripped)

    # Join lines per section
    return {k: "\n".join(v).strip() for k, v in sections.items() if v}

def summarize_sections(sections, max_input_tokens=1500, max_output_tokens=300):
    summarized = {}
    for name, content in sections.items():
        # Skip very short sections
        if len(content.strip().split()) < 10:
            continue

        # Truncate content to avoid token limit issues
        truncated = content.strip().replace("\n", " ")[:4000]

        try:
            summary = summarizer(
                truncated,
                max_length=max_output_tokens,
                min_length=100,
                do_sample=False
            )
            summarized[name] = summary[0]['summary_text']
        except Exception as e:
            print(f"Failed to summarize section '{name}': {e}")

    return summarized

def extract_phrases_from_summaries(summaries, phrases_per_section=20):
    all_phrases = []
    for section, text in summaries.items():
        # Extract keywords/phrases using KeyBERT
        keywords = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 3),
            stop_words='english',
            use_maxsum=True,
            nr_candidates=20, # number of noun phrases returned for each summary
            top_n=phrases_per_section
        )
        section_phrases = [kw for kw, score in keywords]
        all_phrases.extend(section_phrases)

    return all_phrases

def get_top_nodes(phrases, limit=100):
    phrase_counts = Counter(phrases)
    return [phrase for phrase, _ in phrase_counts.most_common(limit)]

def summary_based_create_nodes(paper_text: str, node_limit: int) -> list:
  phrases = extract_phrases_from_summaries(summaries)

  if len(phrases) < node_limit:
    node_limit = len(phrases)

  nodes = get_top_nodes(phrases, node_limit)
  return nodes

## Step 5: Creation of Edges

In [9]:
# Step 5: Create edges
# Note this is a section of the code where there are multiple strategies
# to do this step. Some examples include:
#     - Co-occurence in sentences or paragraphs
#     - Semantic similarity between node phrases
#     - Text Window Co-occurence
#     - Manual/Domain-Specific Rules (custom created)

# The following does co-occurence
def create_edges_by_sentence_coocurrence(nodes, paper_text):
  nlp = spacy.load("en_core_web_sm")
  doc = nlp(paper_text)

  sentences = [sent.text.lower() for sent in doc.sents]
  edge_weights = defaultdict(int)

  for sentence in sentences:
    present_nodes = [node for node in nodes if node in sentence]
    for node1, node2 in itertools.combinations(set(present_nodes), 2):
      edge = tuple(sorted((node1, node2)))
      edge_weights[edge] += 1

  edges = [(a, b, {'weight': w}) for (a, b), w in edge_weights.items()]
  return edges

In [10]:
def create_edges_by_summary_cooccurrence(summaries, nodes):
    edges = defaultdict(int)

    for summary in summaries.values():
        summary_lower = summary.lower()
        present_nodes = [node for node in nodes if node in summary_lower]

        for a, b in combinations(set(present_nodes), 2):
            edges[(a, b)] += 1  # Increment weight if they co-occur

    return [(a, b, {'weight': w}) for (a, b), w in edges.items()]

# edge_counter = create_edges_by_summary_cooccurrence(summaries,nodes)

# Add a constraint for one connection always!

In [11]:
def create_edges_by_embedding_similarity(nodes, threshold=0.5):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(nodes, convert_to_tensor=True)

    edges = []
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            sim = util.cos_sim(embeddings[i], embeddings[j]).item()
            if sim >= threshold:
                edges.append((nodes[i], nodes[j], {'weight': sim}))
    return edges

## Step 6: Populate Graph

In [12]:
# Step 6: Populate graph

def create_graph(nodes, edges):
  graph = nx.Graph()

  for node in nodes:
    graph.add_node(node)

  for node1, node2, weight in edges:
    graph.add_edge(node1, node2, weight=weight)

  print(f"Number of nodes: {len(graph.nodes)}")
  print(f"Number of edges: {len(graph.edges)}\n")

  return graph

## Steps 7 & 8: Write Out And Display Graph

In [40]:
def graph_to_html(graph, path: str, display: bool):
    net = Network(height="750px", width="100%", notebook=True, cdn_resources="in_line")
    net.from_nx(graph)
    # Write HTML with UTF-8 encoding to avoid UnicodeEncodeError on Windows
    html_str = net.generate_html()
    with open(path, "w", encoding="utf-8") as f:
        f.write(html_str)

    if display:
        from IPython.display import HTML
        with open(path, "r", encoding="utf-8") as f:
            html_content = f.read()
        display(HTML(html_content))

In [14]:
# path = "kg_baseline_50nodes.html"
# graph_to_html(baseline_graph, path, True)

In [15]:
# path = "kg_summary_cooccurence_50nodes.html"
# graph_to_html(summary_cooccurence_graph, path, True)

In [16]:
# path = "kg_summary_embedding_similarity_50nodes.html"
# graph_to_html(summary_embedding_similarity_graph, path, False)

In [17]:
# Step 8: Display in notebook
# from IPython.display import HTML

# with open("kg_summary_embedding_similarity_50nodes.html", "r", encoding="utf-8") as f:
#     html_content = f.read()

# HTML(html_content)

# End-to-end code to create a knowledge graph from research papers

In [51]:
paper_path = "pdfs/Transfer learning improves performance in volumetric.pdf"
paper_text = load_paper_text_from_file(paper_path)
print(f"Sample text from paper:\n{paper_text[:500]}...")

Sample text from paper:
In Review, , pp. 1–8
doi:
Transfer learning improves performance in volumetric
electron microscopy organelle segmentation across
tissues
Ronald Xie,1,2,3, Ben Mulcahy,4 Ali Darbandi,4 Sagar Marwah,4 Fez Ali,1 Yuna Lee,1
Gunes Parlakgul,5 Gokhan Hotamisligil,6,7 Bo Wang,2,8,11,12, Sonya MacParland,8,9
Mei Zhen3,4 and Gary D. Bader1,3,4,12,13
1The Donnelly Centre, University of Toronto, Toronto, Ontario, Canada, 2Peter Munk Cardiac Centre and Joint Department of Medical
Imaging, University Health ...


## Nodes

In [52]:
# create nodes using the baseline method

baseline_nodes = baseline_create_nodes(paper_text, 50)
print(baseline_nodes)
len(baseline_nodes)

['the transferred model', 'the performance', 'vem data', 'the pretraining task', '90 minutes', 'model predictions', 'the rat liver dataset', 'a model', 'a target task', '219 serial 7000x7000 pixel images', 'all target tasks', 'upsampling branch', 'the community', 'our\nresults', 'an additional test', 'mitochondria segmentation', 'different domains', 'our model', 'the target task', 'manual labeling', 'transfer\nlearning', 'er segmentation', 'the\ntarget task', 'abundant labels', 'the objects', 'date month year', 'short article title', 'transfer learning', 'vem datasets', 'the exception', 'university health network', 'the resulting segmentations', 'the dataset', 'serial block face', 'both mitochondria', 'deep learning models', 'randomly initialized weights', 'a\nmodel', 'the amount', 'electron microscopy', 'cellular structures', 'each layer', 'the intersection', 'training\ndata', 'nature methods', 'et al.\nfig', 'different tissues', 'the model', 'training data', 'the image volumes']


50

In [ ]:
# create nodes using the summary-based method

summarizer = pipeline("summarization")
kw_model = KeyBERT()
sections = extract_sections(paper_text) # Can clean text here (in-text citations, figs, etc.)
summaries = summarize_sections(sections)
summary_based_nodes = summary_based_create_nodes(paper_text, 50)
print(summary_based_nodes)
print(len(summary_based_nodes))

## Edges

In [54]:
sentence_coocurrence_edges = create_edges_by_sentence_coocurrence(baseline_nodes, paper_text)
summary_cooccurence_edges = create_edges_by_summary_cooccurrence(sections, summary_based_nodes)
embedding_similarity_edges = create_edges_by_embedding_similarity(summary_based_nodes, 0.8)

## Graph

In [55]:
baseline_graph = create_graph(baseline_nodes, sentence_coocurrence_edges)
summary_embedding_similarity_graph = create_graph(summary_based_nodes, embedding_similarity_edges)
summary_cooccurence_graph = create_graph(summary_based_nodes, summary_cooccurence_edges)

Number of nodes: 50
Number of edges: 122

Number of nodes: 50
Number of edges: 85

Number of nodes: 50
Number of edges: 89



## Export Graph to HTML

In [ ]:
graph_to_html(baseline_graph, "kg_baseline_graph.html", False)
graph_to_html(summary_embedding_similarity_graph, "kg_summary_embedding_similarity_50nodes.html", False)
graph_to_html(summary_cooccurence_graph, "kg_summary_cooccurence_50nodes.html", False)